## 1. Persiapan Library

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras
from keras import layers, models

from sklearn.metrics import classification_report, confusion_matrix

## 2. Mengunduh Dataset dari KaggleHub

In [ ]:
%pip install -q kagglehub

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("farzadnekouei/trash-type-image-dataset")
print("Lokasi dataset:", dataset_path)

In [ ]:
for root_dir, sub_dirs, file_list in os.walk(dataset_path):
    print(root_dir, "->", sub_dirs)

## 3. Konfigurasi Parameter Dataset

In [ ]:
DATA_DIR = dataset_path
IMAGE_DIM = (128, 128)
BATCH = 32
RANDOM_SEED = 42

## 4. Membuat Dataset Training dan Validasi

In [ ]:
ds_train = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="training",
    seed=RANDOM_SEED,
    image_size=IMAGE_DIM,
    batch_size=BATCH
)

ds_val = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=RANDOM_SEED,
    image_size=IMAGE_DIM,
    batch_size=BATCH
)

label_names = ds_train.class_names
print("Daftar kelas:", label_names)

## 5. Menampilkan Contoh Gambar dari Dataset

In [ ]:
plt.figure(figsize=(9, 9))
for batch_images, batch_labels in ds_train.take(1):
    for idx in range(9):
        plt.subplot(3, 3, idx + 1)
        plt.imshow(batch_images[idx].numpy().astype("uint8"))
        plt.title(label_names[batch_labels[idx]])
        plt.axis("off")
plt.show()

## 6. Optimasi Pipeline Data

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
ds_train = ds_train.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
ds_val = ds_val.cache().prefetch(buffer_size=AUTOTUNE)

total_classes = len(label_names)
print("Total kelas:", total_classes)

## 7. Membangun Arsitektur Model CNN

In [ ]:
cnn_model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(IMAGE_DIM[0], IMAGE_DIM[1], 3)),

    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(total_classes, activation='softmax')
])

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()
print("Total parameter model:", cnn_model.count_params())

## 8. Melatih Model

In [ ]:
TOTAL_EPOCH = 20

training_history = cnn_model.fit(
    ds_train,
    validation_data=ds_val,
    epochs=TOTAL_EPOCH
)

## 9. Visualisasi Hasil Training

In [ ]:
train_acc = training_history.history['accuracy']
valid_acc = training_history.history['val_accuracy']
train_loss = training_history.history['loss']
valid_loss = training_history.history['val_loss']
epoch_axis = range(TOTAL_EPOCH)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epoch_axis, train_acc, label='Akurasi Training', marker='o')
plt.plot(epoch_axis, valid_acc, label='Akurasi Validasi', marker='o')
plt.legend(loc='lower right')
plt.title('Akurasi Training vs Validasi')

plt.subplot(1, 2, 2)
plt.plot(epoch_axis, train_loss, label='Loss Training', marker='o')
plt.plot(epoch_axis, valid_loss, label='Loss Validasi', marker='o')
plt.legend(loc='upper right')
plt.title('Loss Training vs Validasi')

plt.tight_layout()
plt.show()

## 10. Evaluasi Model pada Data Validasi

In [ ]:
eval_loss, eval_acc = cnn_model.evaluate(ds_val)
print(f"Loss Validasi : {eval_loss:.4f}")
print(f"Akurasi Validasi : {eval_acc * 100:.2f}%")

## 11. Classification Report dan Confusion Matrix

In [ ]:
actual_labels = []
predicted_labels = []

for batch_images, batch_labels in ds_val:
    batch_preds = cnn_model.predict(batch_images, verbose=0)
    predicted_labels.extend(np.argmax(batch_preds, axis=1))
    actual_labels.extend(batch_labels.numpy())

print(classification_report(actual_labels, predicted_labels, target_names=label_names))

conf_mat = confusion_matrix(actual_labels, predicted_labels)
plt.figure(figsize=(8, 6))
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Greens',
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix')
plt.show()

## 12. Contoh Prediksi pada Gambar Validasi

In [ ]:
plt.figure(figsize=(9, 9))
for batch_images, batch_labels in ds_val.take(1):
    batch_preds = cnn_model.predict(batch_images, verbose=0)
    pred_idx = np.argmax(batch_preds, axis=1)
    for idx in range(9):
        plt.subplot(3, 3, idx + 1)
        plt.imshow(batch_images[idx].numpy().astype("uint8"))
        plt.title(f"Aktual: {label_names[batch_labels[idx]]}\nPrediksi: {label_names[pred_idx[idx]]}")
        plt.axis("off")
plt.tight_layout()
plt.show()

## 13. Menyimpan Model

In [ ]:
cnn_model.save("model_klasifikasi_sampah.keras")
print("Model berhasil disimpan.")